# VIGO Python walkthrough

One City, three questions. This notebook builds or opens a City, then works through the analyses most studies need:

1. **Route**: a journey there and back, arrive-by, and alternative journeys.
2. **Matrix**: travel times between groups of places, keeping unreachable pairs visible, then journey and leg tables.
3. **Reach**: where you can get within 15, 30, and 45 minutes, exported as GeoJSON.
4. **Scenario**: an illustrative new service, compared with today's network.
5. **Batches**: a departure-time sweep in parallel, and a batch that keeps every outcome.
6. **A study record** that lets someone reproduce the run.

**Before you start**

- Install VIGO Python with a platform wheel, which includes the engine runtime ([installation](../docs/guide.md#your-first-python-query)).
- Download the two Boston inputs from the [Engine quickstart](https://github.com/hytangs/vigo/blob/main/docs/guide.md#quickstart) into `./data`: `MBTA_GTFS.zip` and `massachusetts-latest.osm.pbf`. If you already built `./boston`, the notebook opens it and skips the build.
- With your own data, edit `GTFS`, `OSM`, `CITY` and `PLACES` in the first cell. The service date defaults to a weekday your feed covers.

Results are scheduled timetable models, not observed journeys. Files go to `./outputs`.

In [ ]:
import datetime as dt
import importlib.util
import json
import os
from collections import Counter
from contextlib import closing
from pathlib import Path

import vigo

# Inputs and the City directory. Environment variables override these for scripted runs.
GTFS = Path(os.environ.get("VIGO_GTFS", "./data/MBTA_GTFS.zip"))
OSM = Path(os.environ.get("VIGO_OSM", "./data/massachusetts-latest.osm.pbf"))
CITY = Path(os.environ.get("VIGO_CITY", "./boston"))
OUTPUT = Path(os.environ.get("VIGO_OUTPUT", "./outputs"))
OUTPUT.mkdir(parents=True, exist_ok=True)

# [longitude, latitude] points inside the City's street coverage.
PLACES = json.loads(os.environ["VIGO_PLACES"]) if "VIGO_PLACES" in os.environ else {
    "harvard": [-71.11902, 42.37334],
    "kendall": [-71.08618, 42.36249],
    "south-station": [-71.05524, 42.35227],
    "copley": [-71.07758, 42.34997],
}
assert {"harvard", "kendall", "south-station", "copley"} <= PLACES.keys(), "PLACES needs all four names"

runtime = vigo.resolve_runtime()
print(f"VIGO Python {vigo.__version__} with Engine {runtime.product_version} (API {runtime.api_version}, {runtime.source})")

## Open the City

A City is a complete prepared directory. Build it **once** from GTFS and OpenStreetMap, then reopen it in every later session. The cell below opens `CITY` when it exists and builds it otherwise. A Boston build takes a few minutes and prints each stage as it goes; larger regions take longer.

The City stays open for the whole notebook so that every query reuses the same resident engine. The last cell closes it.

In [ ]:
if (CITY / "network.json").is_file():
    print(f"Opening the City at {CITY}")
    city = vigo.open(CITY, runtime=runtime)
else:
    missing = [str(path) for path in (GTFS, OSM) if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"No City at {CITY} and no inputs to build one: {missing}. "
                                "Download them as described above, or point GTFS and OSM at your files.")
    print(f"Building a City at {CITY} from {GTFS.name} and {OSM.name}")
    city = vigo.build(CITY, gtfs=GTFS, osm=OSM, runtime=runtime, progress=True)

print(city.name, "revision", city.revision_id, "built", city.built_at)
for kind, sources in city.sources.items():
    print(f"  {kind}: {json.dumps(sources)[:160]}")

# A weekday the timetable covers: the first Tuesday on or after the feed starts.
period = city.service_period()
if "VIGO_SERVICE_DATE" in os.environ:
    SERVICE_DATE = os.environ["VIGO_SERVICE_DATE"]
elif period is None:
    raise ValueError("This City has no scheduled service dates; set VIGO_SERVICE_DATE")
else:
    first = period[0]
    SERVICE_DATE = (first + dt.timedelta(days=(1 - first.weekday()) % 7)).isoformat()
print("timetable", " to ".join(map(str, period)) if period else "undated", "| service date", SERVICE_DATE)

## Route: plan a journey

`status` comes first. `ready` carries a journey; `blocked` is a valid answer meaning no journey satisfies the date, walking limit, transfer cap, and horizon you asked for. Invalid requests and failed operations raise `vigo.VigoError` instead.

Python Results use service-day minutes: `525` is 08:45, and values above 1440 belong to the same service day after midnight.

In [ ]:
def clock(minutes):
    """Service-day minutes as HH:MM; hours can exceed 23 after midnight."""
    if minutes is None:
        return "--:--"
    whole = round(minutes)
    return f"{whole // 60:02d}:{whole % 60:02d}"


def describe(result):
    if result.status != "ready":
        print("No journey under these constraints (blocked).")
        return
    journey = result.value
    print(f"{clock(journey['departMinutes'])} -> {clock(journey['arriveMinutes'])}: "
          f"{result.duration_minutes:.0f} min, {journey['transfers']} transfer(s), "
          f"{journey['walkMinutes']:.0f} min walking, {journey['waitMinutes']:.0f} min waiting")
    for leg in result.legs:
        label = leg.get("routeShortName") or leg["type"]
        print(f"  {clock(leg.get('startMinutes'))}-{clock(leg.get('endMinutes'))}  {label:>8}  "
              f"{leg.get('fromName', '')} -> {leg.get('toName', '')}")
    for warning in result.warnings:
        print("  warning:", warning)


trip = {"service_date": SERVICE_DATE, "max_walk_km": 1.2, "require_transit_ride": True}
outbound = city.route(PLACES["harvard"], PLACES["south-station"], depart_at="08:00", **trip)
describe(outbound)

Arrive-by searches for the **latest** departure that still arrives by the deadline. Reversing the trip is a different question: service, transfers, and walking all change with direction.

In [ ]:
deadline = city.route(PLACES["harvard"], PLACES["south-station"], arrive_by="09:00", **trip)
describe(deadline)

homeward = city.route(PLACES["south-station"], PLACES["harvard"], depart_at="17:30", **trip)
describe(homeward)

outbound.export(OUTPUT / "route.json")
outbound.export(OUTPUT / "route.geojson")

### Alternative journeys

`alternatives=3` asks for more journeys at the requested time. The default `pareto` ranking keeps only real trade-offs between arrival, boardings, and walking. `city.supports(query)` checks the combination against this runtime before you run it.

In [ ]:
query = vigo.Route(PLACES["harvard"], PLACES["south-station"], depart_at="08:00", alternatives=3, **trip)
support = city.supports(query)
if not support.supported:
    print("Not available with this runtime:", support.reason, support.available)
else:
    options = city.run(query)
    describe(options)
    print("search:", options.alternative_search)
    for number, choice in enumerate(options.choices, 1):
        print(f"  choice {number}: {clock(choice.get('departMinutes'))} -> {clock(choice.get('arriveMinutes'))}, "
              f"{choice.get('transfers')} transfer(s), {choice.get('walkMinutes', 0):.0f} min walking")

## Matrix: travel times between groups

Give every point a stable ID so rows stay joinable when the input order changes. Here two origins must reach two destinations **by 09:00**: one reverse search serves each destination.

Count outcomes before aggregating. A completed Matrix can contain `blocked` pairs, and their duration is missing, never zero. For arrive-by, `durationMinutes` runs from the latest departure to the deadline, so it includes any wait at the destination.

In [ ]:
origins = {name: PLACES[name] for name in ("harvard", "kendall")}
destinations = {name: PLACES[name] for name in ("south-station", "copley")}

matrix = city.matrix(
    origins,
    destinations,
    arrive_by="09:00",
    service_date=SERVICE_DATE,
    max_walk_km=1.2,
    max_transfers=3,
    include_journeys=True,
)

print(dict(Counter(row["status"] for row in matrix.iter_rows())))
for row in matrix.iter_rows():
    if row["status"] == "ready":
        answer = f"leave {clock(row['departMinutes'])}, {row['durationMinutes']:.0f} min to the deadline"
    else:
        answer = "no journey"
    print(f"  {row['originId']:>8} -> {row['destinationId']:<14} {answer}")

matrix.export(OUTPUT / "matrix.json")
matrix.export(OUTPUT / "matrix.csv")

### Journey and leg tables

`include_journeys=True` above kept each pair's itinerary, so the Matrix can be decomposed into one row per journey and one row per leg. `export_tables` writes `journeys.csv`, `legs.csv`, and `record.json`; blocked pairs stay in the journey table with `hasJourney` false. pandas is optional.

In [ ]:
tables = matrix.export_tables(OUTPUT / "matrix-tables")
print("wrote", sorted(path.name for path in tables.iterdir()))

legs = list(matrix.iter_legs())
print("legs by type:", dict(Counter(leg["type"] for leg in legs)))

if importlib.util.find_spec("pandas"):
    import pandas as pd

    journeys = pd.DataFrame(matrix.iter_journeys())
    print(journeys[["originId", "destinationId", "status", "hasJourney", "durationMinutes", "transfers", "walkMinutes"]]
          .to_string(index=False))
else:
    print("Install pandas to explore these records as DataFrames.")

## Reach: where can you get?

Reach computes a travel-time surface from one origin. The extent radius sets the analysis grid, not a trip-length limit; keep origin, date, time, walking assumptions, and grid identical whenever you compare surfaces. Values are minutes, and `None` means unreached.

Reach measures modeled travel time. It does not count jobs, people, or other opportunities.

In [ ]:
reach_query = vigo.Reach(
    PLACES["harvard"],
    depart_at="08:00",
    service_date=SERVICE_DATE,
    cutoffs_minutes=(15, 30, 45),
    max_walk_km=1.2,
    extent_radius_km=6,
    raster_size=96,
)
baseline = city.run(reach_query)

values = baseline.value["values"]
for cutoff in reach_query.cutoffs_minutes:
    reached = sum(value is not None and value <= cutoff for value in values)
    print(f"within {cutoff:>2} min: {reached:6,d} of {len(values):,d} cells")

baseline.export(OUTPUT / "reach.json")
baseline.export(OUTPUT / "reach.geojson")

## Scenario: test a service proposal

A Scenario changes the service one query sees, tied to this City revision. This one adds a ten-minute crosstown service between Harvard and Copley from 07:00 to 10:00 at an assumed 20 km/h. The numbers are **illustrative assumptions**, not an operating plan.

Planned service applies to Reach; `supports()` explains why a Route cannot use it.

In [ ]:
proposal = city.scenario(
    "Illustrative crosstown",
    services=[{
        "operation": "add",
        "name": "Crosstown",
        "stops": [
            {"label": "Harvard", "coordinate": PLACES["harvard"]},
            {"label": "Copley", "coordinate": PLACES["copley"]},
        ],
        "headwayMinutes": 10,
        "startMinutes": 420,
        "endMinutes": 600,
        "averageSpeedKph": 20,
    }],
)
print("Reach:", proposal.supports(reach_query))
print("Route:", proposal.supports(vigo.Route(PLACES["harvard"], PLACES["copley"], depart_at="08:00", service_date=SERVICE_DATE)))

after = proposal.run(reach_query)
change = baseline.compare(after)
print(json.dumps(change.value, indent=2))

after.export(OUTPUT / "reach-proposal.geojson")
change.export(OUTPUT / "reach-change.json")

Read all the counts together: `fasterCells` and `slowerCells` describe cells reachable in both runs, and `meanChangeMinutes` averages only those. Newly reachable and no-longer-reachable cells are counted separately. A larger area is not, by itself, a benefit to people.

## Batches

### A departure-time sweep, in parallel

Independent queries can overlap. `iter_run(..., workers=2)` yields complete Results in input order while bounding memory; each worker owns a resident engine process, so measure before raising `workers`.

In [ ]:
departures = [f"{hour:02d}:{minute:02d}" for hour in (7, 8) for minute in (0, 15, 30, 45)] + ["09:00"]
sweep = [
    vigo.Route(PLACES["harvard"], PLACES["south-station"], depart_at=time, **trip)
    for time in departures
]
with closing(city.iter_run(sweep, workers=2)) as results:
    for time, result in zip(departures, results):
        if result.status == "ready":
            journey = result.value
            print(f"leave {time}: arrive {clock(journey['arriveMinutes'])} ({result.duration_minutes:.0f} min, "
                  f"{journey['transfers']} transfer(s))")
        else:
            print(f"leave {time}: no journey")

### A batch that keeps every outcome

`city.run([...])` and `iter_run` stop at the first error. When a study must keep every success, blocked answer, and failure, loop explicitly. Opening the City happens outside the loop: a setup failure is not one blocked journey per request.

In [ ]:
requests = {
    f"{origin}-to-{destination}": vigo.Route(PLACES[origin], PLACES[destination], depart_at="08:00",
                                             service_date=SERVICE_DATE, max_walk_km=1.2)
    for origin in PLACES
    for destination in PLACES
    if origin != destination
}
batch = OUTPUT / "batch"
batch.mkdir(exist_ok=True)
outcomes = []
for request_id, request in requests.items():
    try:
        result = city.run(request)
    except vigo.VigoError as error:
        outcomes.append({"id": request_id, "outcome": "query_error", "error": f"{type(error).__name__}: {error}"})
        continue
    result.export(batch / f"{request_id}.json")
    outcomes.append({"id": request_id, "outcome": result.status, "durationMinutes": result.duration_minutes})

(batch / "outcomes.json").write_text(json.dumps(outcomes, indent=2) + "\n", encoding="utf-8")
print(dict(Counter(item["outcome"] for item in outcomes)))

## Keep a reproducible record

Store the inputs that produced these files beside them: package and engine versions, the City revision and sources, the date, the places, and each Result's normalized query, status, warnings, and timing. Keep the City directory and raw GTFS/OSM files too; the record identifies them but does not contain them.

In [ ]:
record = {
    "vigoPython": vigo.__version__,
    "engine": {"version": runtime.product_version, "api": runtime.api_version, "source": runtime.source},
    "city": {"name": city.name, "revisionId": city.revision_id, "builtAt": city.built_at, "sources": city.sources},
    "serviceDate": SERVICE_DATE,
    "places": PLACES,
    "results": {
        "route": outbound.record,
        "matrix": matrix.record,
        "reach": baseline.record,
        "reachProposal": after.record,
        "batch": outcomes,
    },
}
(OUTPUT / "study.json").write_text(json.dumps(record, indent=2, default=str) + "\n", encoding="utf-8")
print("wrote", sorted(path.name for path in OUTPUT.iterdir()))

In [ ]:
city.close()

## Next steps

- [Workflows](../docs/guide.md#workflows) and [Results](../docs/guide.md#results) explain each pattern and how to read its fields.
- The [API reference](../docs/reference.md) lists every argument and default.
- [`decompose_result.py`](decompose_result.py) turns a saved Route or Matrix JSON into the same journey and leg tables from the command line.